# Parallelization Testing Workbench


This is a showcase of our parallelization pipeline, including unit tests, devised by Dason. If you have any questions about the content of this notebook, feel free to book a meeting with us!

To begin, we wll install dependencies and print out our system specifications. I am currently running this on my personal laptop, not on the development server for several reasons.

- it saves me deubgging time
- it allows me to quickly edit and run tests
- I know broadly the capabilities of my machine, and can compare it to prior MATLAB results

Speaking of which, comparisons between MATLAB speed and Python speed can be found in the "Older Results" folder, though those are based on matrix computations and not any Python models (at the time, we didn't have converted Python test models ready yet).


In [5]:
# get requirements
%pip install -q -r "requirements.txt"

# get logging untilites
%pip install -q psutil py-cpuinfo

# print out system hardware specifications
import psutil
import platform
import cpuinfo

def get_size(bytes, suffix="B"):
    """Scale bytes to its proper format (e.g., GB, MB)"""
    factor = 1024
    for unit in ["", "K", "M", "G", "T", "P"]:
        if bytes < factor:
            return f"{bytes:.2f}{unit}{suffix}"
        bytes /= factor

print("="*20, "System Information", "="*20)
uname = platform.uname()
print(f"System: {uname.system}")
print(f"Release: {uname.release}")
print(f"Machine: {uname.machine}")

# CPU Information
print("\n" + "="*20, "CPU Info", "="*20)
# Use cpuinfo for the brand name
print(f"Processor: {cpuinfo.get_cpu_info()['brand_raw']}")
print(f"Physical cores: {psutil.cpu_count(logical=False)}")
cpufreq = psutil.cpu_freq()
print(f"Max Frequency: {cpufreq.max:.2f}Mhz")

# Memory Information
print("\n" + "="*20, "Memory Info", "="*20)
svmem = psutil.virtual_memory()
print(f"Total RAM: {get_size(svmem.total)}")
print(f"Available RAM: {get_size(svmem.available)}")

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
==================== System Information ====================
System: Windows
Release: 11
Machine: AMD64

==================== CPU Info ====================
Processor: 11th Gen Intel(R) Core(TM) i7-11390H @ 3.40GHz
Physical cores: 4
Max Frequency: 2918.00Mhz

==================== Memory Info ====================
Total RAM: 31.75GB
Available RAM: 17.22GB


Note that our server also contains 4 cores, but far less total available memory at 12GB.
You will notice a theme of RAM being the parallelization bottleneck, and we have taken this into account for the future so that later resource allocations include more dedicated memory.


### Explanation of Parallelization Setup


In our V&V document, we wrote:

- "The Application server is the main entry point for all client requests. It routes all HTTPS requests to appropriate modules and communicates with the Scheduler to assign tasks to clusters."

In the final implementation, this is true, but a few compromises were made. The application server does "communicate" with the Scheduler, but rather than listening on stand-by, it only passes along data to it via the standard evaluation call process. The scheduler is unable to modify the behavior of the server, such as dynamically telling the server to cease accepting submissions due to an issue with the most recent submission. Therefore, the server will pass the submission to the scheduler (after the submission processing and verification system) and then the scheduler will send back either the complete result, or an error message.

The data travels as follows:

- User submits a model (front end), sent to back end
- App server handles submission, begins the processing and validation process by calling "process_submission.py"
- "process_submission.py" calls various utilities to validate submission and create figures
- submission -> validation -> "batch_scheduler" -> generate graphs
- previously, email would also be part of this pipeline, but we refactored it to be a server responsibility

You can view all these components in SETool_Python, which is pulled from the most recently deployed version of our app.

The scheduler is "batch_scheduler.py," it is responsible for determining which clusters to use for any given user submission, and then scheduling them through coiled.io (third party cloud orchestration platform, we have an account with cost factored in). We are also connected to our Google Cloud Compute and Canada Compute clusters through the service.

**In order to run these results yourself, you need to integrate your Python environment wth Coiled and link cloud computing resources. To see a full demo, please book a meeting with us.**

The complexity grader is "complexity_grader.py," it is responsible for grading a Python model on heuristics we have developed. It returns a score from 0-2, in ascending complexity. We called these "P0-P2" scores, which stand for "Necessary Parallelization Level." These levels are broadly categorized as follows:

- P0 (Local), usually indicates a linear or linearithmic algorithm, likely deterministic or simply just one function. These include non-stochastic coloumb counters, for example. Typically, the orchestration cost for these is not worth the time and credits compared to running them locally.

- P1 (Parallelized), usually indicates a O(n^2) algorithm, where n is the size of the test dataset. These tend to be well suited for distributed computing if there is a large queue or back-to-back submissions. However, they are still not justified if we only intend to run them once, then deallocate the VM resources.

- P2 (Parallelized with GPU), usually indicates a neural net (LSTM) or highly complex algorithm of a similar scale. These are run with the most amount of workers with dedicated NVIDIA T4 GPUS, and are exorbitantly expensive to run. Most of our available credits were consumed running tests of this class, and to downsize, we have temporarily **disabled** the GPUs allocated to these workers, so the performance here is much worse than typical for these submissions. When GPUs are enabled, they can be justified to run and be killed immediately, as the startup cost is eclipsed by the total runtime cost.

These measures also have corresponding MATLAB complexities returned by the evaluator! They range from 0-9, and it returns a range with width of 3-4, such that P0 models are roughly 0,1,2,3 complex, P1 are roughly 4,5,6, and P2 are roughly 7,8,9. These grades are provided **after** each model is completely evaluated, so we will be using this metric later to demonstrate the correctness of our complexiy grader!


In [1]:
# we now import everything we need
from SETool_Python import *

### Parallelization Lifecycle


According to our V&V document, the following was our design for our parallelization setup:

- This component supports parallelization, enables distributed computing for load balancing, and arranging the order requests that are handled for optimal speed and user satisfaction (submission queue). Communicates directly with Computing Clusters & Application Server and is partially built-in to the Battery SOC Evaluation Algorithm (Python) with Dask Dataframes used as the unified parallelized data structure, replacing NumPy arrays, Python lists, and any parallelized matrix representations.

- Listens to main Application Server: receives requests with associated data and assigns tasks to Computing Cluster “optimally”

- Determines whether to use local computing (no internet access is the primary factor but also includes assessment of the busyness of the available computing clusters) to handle submission.

- Splits up data to be evaluated in parallel, handled automatically through Dask Dataframes paired with Coiled management. We have rewritten the Python code to allow for simple parallelization through Dask.

- Queues tasks and assigns them to the correct cluster threads or local threads. Thread assignment logic handled by Coiled, but the queue and assignment is custom built.

- Offers load-balancing to guarantee throughput and user satisfaction. Communicates to Application Server information about the availability of compute power and monthly budget. This includes choices regarding: When to start/stop a cluster, What CPU/GPU options to use, How many computing resources to use in parallel for the current task in the queue, given final budget, When to discard tasks

We were able to **achieve all of the above,** but we made notable compromises as a trade-off between latency and throughput. From the academic literature, we have essentially implemented Data-level parallelism, but failed to design an optimal "queuing" system at the task-level:

1. Task-Level Parallelism (TLP)

Coarse-Grained Parallelism / High-Throughput Computing (HTC)

This is our first, intended approach: using a queue to send different "submissions" (tasks) to different workers. Each CPU or worker handles a completely independent piece of work from start to finish.

    How it works: You treat each submission as a "Black Box." The system doesn't care what is happening inside the task; it only cares about finding an available CPU to run it.

    The Goal is Throughput. You want to finish 1,000 tasks in the shortest time possible, even if any single task still takes the same amount of time to complete as it would locally.

    Best for: problems where tasks don't need to talk to each other, such as processing 100 different battery test cycles simultaneously.

    Data Principle: MIMD (Multiple Instruction, Multiple Data). Each processor is executing different instructions on different pieces of data at the same time.

2. Data-Level Parallelism (DLP)

Fine-Grained Parallelism / Strong Scaling

Due to complexities involved with scheduling simultaneus submissions, we decided to simplify to a linear submission: we evaluate each submission, one by one, in the order they were recieved. We use Dask to run one submission at a time but breaking its internal data (the arrays/matrices, as dask dataframes) apart so that every available CPU works on a small slice of that single submission.

    How it works: You "decompose" the data. If you have a large NumPy array, CPU 1 calculates the first 25%, CPU 2 the next 25%, and so on. In your specific mention of "thread switching," this often involves Concurrency, where the OS manages multiple threads on a single node to keep the CPU cores saturated while waiting for I/O or memory access.

    The Goal is Latency. You want to make this specific submission finish as fast as possible.

    Best for: Large-scale simulations, heavy matrix math, or training a single large machine learning model where the data is too big for one CPU to handle efficiently. This is very good for our P2-level submissions.

    Key Academic Term: SIMD (Single Instruction, Multiple Data) or SPMD (Single Program, Multiple Data). You are executing the same logic (your model's predict function) across many pieces of data simultaneously.

Our original ambition for our scheduler would be that it could handle both types of parallelism: we can choose to queue, dequeue, and reorder tasks between different CPUs. We would also break down each task's data to separate CPUs, the most efficient way would be to schedule the same dataframe to cores on the same server, as is typically done on large cloud setups. However, given our lack of manpower (just me!) it was simpler to adopt the Data-Level only approach.

The following tests comprise of two sections: UNIT TESTS will be conducted using our own testing suite (we have detailed logs of the performance of each run, outputted in performance.log). PERFORMANCE TESTS will be conducted with PyTest


In [2]:
# get pytest
%pip -q install pytest

Note: you may need to restart the kernel to use updated packages.


### Preamble about the models used in these tests

Very briefly, we have produced several models directly converted or otherwise derived from MATLAB models. You can find them in the Python Models folder:

- Coloumb Counter Basic (Assigned to User 1) is linear, simple, and runs best as P0
- Coluumb Counter Enhance (Assigned to User 2) involves an additional decay calculation which does a table lookup, requiring more memory, and runss best as P0
- Coloumb Counter Stochastic O(n^2) (Assigned to User 3) is a dummy approximation of a proprietary algorithm **which uses a nested for loop over the voltage, charge, and temperature fluxuations at this time instant.** This is NOT a model that has been verified by our supervisor, and essentially only does "busy work" so that we can demonstrate the robustness of our system. It runs best as a P1 task.
- Extended Kalman Filter is an O(n^2) task (Assigned to User 4), but the diagonalizations required to translate Matlab to Python technically make it O(n^3), once having been diagonalized, no loops are then required. This deceptively fools our complexity grader into thinking it it is P0, and is a great example of a P1 task that "sneaks" into local processing.

We do not provide any LSTM models because our supervisor has not provided them to us yet (as you can imagine, they take a while to convert and verify), though we have tried simple FNNs of our own making. They eat up so much compute cost, however, that we will omit them from comparison. Overall, this suite is not designed to fully showcase our P2, as we have disabled GPU at the moment and do not use a fittingly complicated model.


### Natively Implemented Unit Tests


Methodology here consists of using psutil to get the snapshots of the system over 100 miliseconds for the duration of the run.

Local runs start counting immediately, whereas parallelized runs start counting the moment after we send over the cluster initialization instructions. This therefore **includes the startup latency to spin up existing VM instances!**

Additionally, we start the process as if we are submitting a user model from scratch, so all steps, including verification and graphing, are included in each run, though only the time it takes to complete each evaluation is logged.

The cost for each call must be fetched from coiled.io, and is included manually thereafter each test:


In [6]:
import os
current_dir = os.getcwd()
path = os.path.join(current_dir, "SETool_Python")
os.chdir(path)

In [5]:
# LOCAL TESTS
%run standardized_evaluation_tool.py "user1" "P0"

Loading test data...
Starting the blind modeling tool...
Zip extracted successfully.
Model file found!
Starting validation...
Starting validation process...
Validation passed!
Running test cycles...
Generating figures...
Calculating scores...
Final weighted score: 5669.9283


{"error": false, "results_path": "w:\\Academia\\McMaster University\\Year 4\\00 Capstone\\Battery-Algorithm-Standardized-Testing-Tool\\Parallelization Workbench\\SETool_Python\\user1\\results\\results.zip", "Weighted_Error": 5669.928, "Test_Scores": [4966.925, 3696.596, 5390.368, 5177.895, 1174.117, 3696.596, 6245.049, 8751.939, 4883.597, 5133.581, 538.355, 185.912, 912.14, 1338.712, 1790.077, 2279.506, 10180.346, 11126.333], "All_Drive_Cycles_Average_RMSE": 4966.925, "All_Drive_Cycles_Average_MAE": 4966.923, "All_Drive_Cycles_Average_MAXE": 4968.669, "Complexity": "0,1,2"}


Results written to: user1\results\results.zip


In [ ]:
%run standardized_evaluation_tool.py "user2" "P0"

Loading test data...
Starting the blind modeling tool...
Zip extracted successfully.
Model file found!
Starting validation...
Starting validation process...
Validation passed!
Running test cycles...
Generating figures...
Calculating scores...
Final weighted score: 30.8117


{"error": false, "results_path": "w:\\Academia\\McMaster University\\Year 4\\00 Capstone\\Battery-Algorithm-Standardized-Testing-Tool\\Parallelization Workbench\\SETool_Python\\user2\\results\\results.zip", "Weighted_Error": 30.812, "Test_Scores": [35.205, 20.254, 40.189, 10.369, 43.003, 20.254, 35.145, 42.418, 26.381, 52.854, 50.715, 43.259, 50.0, 38.099, 31.794, 44.151, 19.002, 30.66], "All_Drive_Cycles_Average_RMSE": 35.205, "All_Drive_Cycles_Average_MAE": 31.158, "All_Drive_Cycles_Average_MAXE": 71.997, "Complexity": "1,2,3"}


Results written to: user2\results\results.zip


In [7]:
%run standardized_evaluation_tool.py "user3" "P0"

Loading test data...
Starting the blind modeling tool...
Zip extracted successfully.
Model file found!
Starting validation...
Starting validation process...
Validation passed!
Running test cycles...
Generating figures...
Calculating scores...
Final weighted score: 30.8117


{"error": false, "results_path": "w:\\Academia\\McMaster University\\Year 4\\00 Capstone\\Battery-Algorithm-Standardized-Testing-Tool\\Parallelization Workbench\\SETool_Python\\user3\\results\\results.zip", "Weighted_Error": 30.812, "Test_Scores": [35.205, 20.254, 40.189, 10.369, 43.003, 20.254, 35.145, 42.418, 26.381, 52.854, 50.715, 43.259, 50.0, 38.099, 31.794, 44.151, 19.002, 30.66], "All_Drive_Cycles_Average_RMSE": 35.205, "All_Drive_Cycles_Average_MAE": 31.158, "All_Drive_Cycles_Average_MAXE": 71.997, "Complexity": "5,6,7"}


Results written to: user3\results\results.zip


In [8]:
%run standardized_evaluation_tool.py "user4" "P0"

Loading test data...
Starting the blind modeling tool...
Zip extracted successfully.
Model file found!
Starting validation...
Starting validation process...
Validation passed!
Running test cycles...
Generating figures...
Calculating scores...
Final weighted score: 20.0939


{"error": false, "results_path": "w:\\Academia\\McMaster University\\Year 4\\00 Capstone\\Battery-Algorithm-Standardized-Testing-Tool\\Parallelization Workbench\\SETool_Python\\user4\\results\\results.zip", "Weighted_Error": 20.094, "Test_Scores": [22.201, 22.133, 22.224, 2.687, 23.051, 22.133, 20.817, 22.805, 19.188, 28.227, 37.294, 30.362, 21.991, 18.469, 15.303, 14.886, 17.36, 22.15], "All_Drive_Cycles_Average_RMSE": 22.201, "All_Drive_Cycles_Average_MAE": 19.533, "All_Drive_Cycles_Average_MAXE": 38.375, "Complexity": "6,7,8"}


Results written to: user4\results\results.zip


Great! Now we can inspect performance.log for the local results, but before doing that, let's try various results running on P1-P3, so we have something to compare our parallelization to:

**NOTE** for the following, we run them such that each container is destroyed before starting (we wait 1-2 minutes per run), therefore, each run involves completely allocating and de-allocating resources, spinning up VMs from scratch!


In [4]:
# DISTRIBUTED TESTS, on WINDOWS
# this enables fast updates if we need to change servers
%load_ext autoreload
%autoreload 2

In [3]:
%run standardized_evaluation_tool.py "user1" "P1"

Loading test data...
Starting the blind modeling tool...
Zip extracted successfully.
Model file found!
Starting validation...
Starting validation process...
Validation passed!
Running test cycles...
Scheduler Activated - Processing mode P1
Spinning up a cluster...
[2026-04-07 15:05:02,112][INFO    ][coiled] Fetching latest package priorities...
[2026-04-07 15:05:02,124][INFO    ][coiled.package_sync] Resolving your local .venv Python environment...
[2026-04-07 15:05:02,292][INFO    ][coiled.package_sync] Scanning 104 python packages...
[2026-04-07 15:05:02,680][INFO    ][coiled] Running pip check...
[2026-04-07 15:05:03,796][INFO    ][coiled] Validating environment...
[2026-04-07 15:05:06,959][INFO    ][coiled] Creating wheel for W:\Academia\McMaster University\Year 4\00 Capstone\Battery-Algorithm-Standardized-Testing-Tool\Parallelization Workbench\SETool_Python...
[2026-04-07 15:05:07,418][INFO    ][coiled] Creating wheel for W:\Academia\McMaster University\Year 4\00 Capstone\Battery-

{"error": false, "results_path": "w:\\Academia\\McMaster University\\Year 4\\00 Capstone\\Battery-Algorithm-Standardized-Testing-Tool\\Parallelization Workbench\\SETool_Python\\user1\\results\\results.zip", "Weighted_Error": 3648.227, "Test_Scores": [4963.214, 3692.879, 5386.66, 5174.178, 1170.424, 3692.879, 6241.332, 8748.223, 4879.889, 5129.864, 534.638, 182.341, 908.423, 1334.995, 1786.361, 2275.789, 109.65, 1009.7], "All_Drive_Cycles_Average_RMSE": 4963.214, "All_Drive_Cycles_Average_MAE": 4963.213, "All_Drive_Cycles_Average_MAXE": 4964.952, "Complexity": "0,1,2"}


Results written to: user1\results\results.zip


In [ ]:
# %run standardized_evaluation_tool.py "user2" "P1"
# skipping because the result is near-identical to the first example

In [4]:
%run standardized_evaluation_tool.py "user3" "P1"

Loading test data...
Starting the blind modeling tool...
Zip extracted successfully.
Model file found!
Starting validation...
Starting validation process...
Validation passed!
Running test cycles...
Scheduler Activated - Processing mode P1
Spinning up a cluster...
[2026-04-07 15:10:18,625][INFO    ][coiled] Fetching latest package priorities...
[2026-04-07 15:10:18,627][INFO    ][coiled.package_sync] Resolving your local .venv Python environment...
[2026-04-07 15:10:18,781][INFO    ][coiled.package_sync] Scanning 104 python packages...
[2026-04-07 15:10:19,063][INFO    ][coiled] Running pip check...
[2026-04-07 15:10:19,835][INFO    ][coiled] Validating environment...
[2026-04-07 15:10:22,581][INFO    ][coiled] Creating wheel for W:\Academia\McMaster University\Year 4\00 Capstone\Battery-Algorithm-Standardized-Testing-Tool\Parallelization Workbench\SETool_Python...
[2026-04-07 15:10:22,585][INFO    ][coiled] Creating wheel for W:\Academia\McMaster University\Year 4\00 Capstone\Battery-

{"error": false, "results_path": "w:\\Academia\\McMaster University\\Year 4\\00 Capstone\\Battery-Algorithm-Standardized-Testing-Tool\\Parallelization Workbench\\SETool_Python\\user3\\results\\results.zip", "Weighted_Error": 30.769, "Test_Scores": [35.152, 20.254, 40.118, 10.369, 42.792, 20.254, 35.145, 42.418, 26.302, 52.854, 50.715, 41.995, 50.0, 38.099, 31.794, 44.151, 19.002, 30.66], "All_Drive_Cycles_Average_RMSE": 35.152, "All_Drive_Cycles_Average_MAE": 31.125, "All_Drive_Cycles_Average_MAXE": 71.502, "Complexity": "6,7,8"}


Results written to: user3\results\results.zip


In [ ]:
%run standardized_evaluation_tool.py "user3" "P2"
# P2 shaves off a minute, using 4 instead of 2

Loading test data...
Starting the blind modeling tool...
Zip extracted successfully.
Model file found!
Starting validation...
Starting validation process...
Validation passed!
Running test cycles...
Scheduler Activated - Processing mode P2
Spinning up a cluster with GPUs...
[2026-04-07 16:40:51,681][INFO    ][coiled] Fetching latest package priorities...
[2026-04-07 16:40:51,682][INFO    ][coiled.package_sync] Resolving your local .venv Python environment...
[2026-04-07 16:40:51,840][INFO    ][coiled.package_sync] Scanning 104 python packages...
[2026-04-07 16:40:52,093][INFO    ][coiled] Running pip check...
[2026-04-07 16:40:52,791][INFO    ][coiled] Validating environment...
[2026-04-07 16:40:53,534][INFO    ][coiled] Creating wheel for W:\Academia\McMaster University\Year 4\00 Capstone\Battery-Algorithm-Standardized-Testing-Tool\Parallelization Workbench\SETool_Python...
[2026-04-07 16:40:53,538][INFO    ][coiled] Creating wheel for W:\Academia\McMaster University\Year 4\00 Capston

{"error": false, "results_path": "w:\\Academia\\McMaster University\\Year 4\\00 Capstone\\Battery-Algorithm-Standardized-Testing-Tool\\Parallelization Workbench\\SETool_Python\\user3\\results\\results.zip", "Weighted_Error": 30.769, "Test_Scores": [35.152, 20.254, 40.118, 10.369, 42.792, 20.254, 35.145, 42.418, 26.302, 52.854, 50.715, 41.995, 50.0, 38.099, 31.794, 44.151, 19.002, 30.66], "All_Drive_Cycles_Average_RMSE": 35.152, "All_Drive_Cycles_Average_MAE": 31.125, "All_Drive_Cycles_Average_MAXE": 71.502, "Complexity": "6,7,8"}


Results written to: user3\results\results.zip


In [7]:
%run standardized_evaluation_tool.py "user4" "P2"
# I'm forcing it to use 6, get the maximum juice for our squeeze!

Loading test data...
Starting the blind modeling tool...
Zip extracted successfully.
Model file found!
Starting validation...
Starting validation process...
Validation passed!
Running test cycles...
Scheduler Activated - Processing mode P2
Spinning up a cluster with GPUs...
[2026-04-07 16:57:31,509][INFO    ][coiled] Fetching latest package priorities...
[2026-04-07 16:57:31,510][INFO    ][coiled.package_sync] Resolving your local .venv Python environment...
[2026-04-07 16:57:31,675][INFO    ][coiled.package_sync] Scanning 104 python packages...
[2026-04-07 16:57:31,979][INFO    ][coiled] Running pip check...
[2026-04-07 16:57:32,671][INFO    ][coiled] Validating environment...
[2026-04-07 16:57:33,429][INFO    ][coiled] Creating wheel for W:\Academia\McMaster University\Year 4\00 Capstone\Battery-Algorithm-Standardized-Testing-Tool\Parallelization Workbench\SETool_Python...
[2026-04-07 16:57:33,435][INFO    ][coiled] Creating wheel for W:\Academia\McMaster University\Year 4\00 Capston

{"error": false, "results_path": "w:\\Academia\\McMaster University\\Year 4\\00 Capstone\\Battery-Algorithm-Standardized-Testing-Tool\\Parallelization Workbench\\SETool_Python\\user4\\results\\results.zip", "Weighted_Error": 35.363, "Test_Scores": [43.012, 43.302, 42.915, 8.827, 43.266, 43.302, 42.145, 43.333, 39.033, 50.969, 50.399, 45.368, 43.879, 41.475, 39.431, 39.044, 17.341, 22.141], "All_Drive_Cycles_Average_RMSE": 43.012, "All_Drive_Cycles_Average_MAE": 38.672, "All_Drive_Cycles_Average_MAXE": 66.657, "Complexity": "7,8,9"}


Results written to: user4\results\results.zip


### Analysis of these results


Now that we've finished getting the results stored in our log file, we can retrieve and inspect the log, like so:


In [8]:
def print_log_file(file_path):
    try:
        with open(file_path, 'r') as file:
            for line in file:
                print(line, end='')
    except FileNotFoundError:
        print(f"File not found: {file_path}")

print_log_file("performance.log")


User 1 Performance Metrics, report generated for LOCAL CPU
Evaluated with psutil, threading, and time modules.
--------------------
   Process Name:               python.exe
   avg. CPU utilization:       96.025%
   avg. memory utilization:    212.22%
   elapsed time:               44.582706451416016 s
   est. CPU usage time:        42.8107171188315 s
   est. memory usage time:     94.61323296483118 s
--------------------

User 2 Performance Metrics, report generated for LOCAL CPU
Evaluated with psutil, threading, and time modules.
--------------------
   Process Name:               python.exe
   avg. CPU utilization:       96.861%
   avg. memory utilization:    254.78%
   elapsed time:               52.477014780044556 s
   est. CPU usage time:        50.829992540740356 s
   est. memory usage time:     133.7011842944021 s
--------------------

User 3 Performance Metrics, report generated for LOCAL CPU
Evaluated with psutil, threading, and time modules.
--------------------
   Process 

We will examine a few things: for models where we ran both a P0 or P1/P2, we will give a comparison of how much the parallelization freed up the CPU and memory.

We care about Latency, and Load Time expressed as a percentile of total elapsed time.
In our preliminary tests, we verified that a 4x better time performance against MATLAB is achievable in Python. Now, we compare against the fully-implemented levels of parallelization.

We will compare by the following:

- **Memory Usage**, take P0 v.s. P1/P2 and determine the ratio of improvement or latency, which we will call the **memory load ratio.** P1 or P2 / P0
- **CPU Usage,**, take P0 v.s. P1/P2 and determine the ratio of improvement or latency, which we will call the **CPU load ratio.** P1 or P2 / P0
- **Elapsed Time,** similar to above this measures the **Task Elapsed Time Differential** P1 or P2 - P0

We will use these to conclude about the efficacy of parallelization, and we will also display the total cost it took to compute the above.


In [42]:
# Copied over the results from the log
# Format: [CPU%, Mem%, Elapsed_s, Est_CPU_s, Est_Mem_s]

metrics = [
    "avg. CPU utilization (%)",
    "avg. memory utilization (%)",
    "elapsed time (s)",
    "est. CPU usage time (s)",
    "est. memory usage time (s)"
]

local_results = [
    [96.025, 212.22, 44.582706451416016, 42.8107171188315, 94.61323296483118],   # User 1
    [96.861, 254.78, 52.477014780044556, 50.829992540740356, 133.7011842944021], # User 2
    [96.932, 227.199, 291.47542786598206, 282.5318312234013, 662.228280728223],  # User 3
    [95.26, 272.633, 657.4624843597412, 626.3016878838379, 1792.4608586530067]   # User 4
]

# Format: [CPU%, Mem%, Elapsed_s, Est_CPU_s, Est_Mem_s]
p1_results = [
    [3.244, 223.779, 96.26187586784363, 3.122532596572073, 215.41348035575965],  # User 1
    [0.977, 258.45, 420.8716387748718, 4.11193303171554, 1087.7431172291917]    # User 3
]

# Format: [CPU%, Mem%, Elapsed_s, Est_CPU_s, Est_Mem_s]
p2_results = [
    [1.143, 354.807, 340.13210821151733, 3.88709459644865, 1206.8123168902632],  # User 3
    [0.771, 394.205, 878.9472227096558, 6.774798765972673, 3464.8512801538577]   # User 4
]

# comparing model 1 local vs para P1
print("User 1 with Basic CCounter")
for i in range(5):
    metric = round(p1_results[0][i] / local_results[0][i], 3)

    print(f"P1 / P0 = {metric} in terms of {metrics[i]}")
print("The cost to run P1 was 0.01 CAD")
print()
# comparing model 3 local vs para P1
print("User 3 with Stochastic CCounter")
for i in range(5):
    metric = round(p1_results[1][i] / local_results[2][i], 3)

    print(f"P1 / P0 = {metric} in terms of {metrics[i]}")
print("The cost to run P1 was 0.05 CAD")
print()
# comparing model 4 local vs para P2
print("User 4 with Kalmann")
for i in range(5):
    metric = round(p2_results[1][i] / local_results[3][i], 3)

    print(f"P1 / P0 = {metric} in terms of {metrics[i]}")
print("The cost to run P2 was 0.16 CAD")

print("\n\nFindings:")
avg_mem = round((1.054 +  1.138 + 1.446) / 3,3)
avg_cpu = round(3/(0.034 + 0.01 + 0.008), 3)
print(f"We verify that typically, parallelization is bottle-necked by memory.\nWe are able to trade an average of {avg_mem} times memory utilization compared to a local model, \nin exchange for roughly {avg_cpu} times free CPU utilization")
print()
print("Elapsed time of both CPU and memory follow a similar trend. Elapsed time ratio is shown to decrease towards 1.0 for more complicated models.")
print()
cost = round((0.01 + 0.05 + 0.16)/3, 3)
time = round((2.159 + 1.444 + 1.337)/3, 3)
print(f"It costs roughly {cost}, and prolongs the execution of a model by {time}")

User 1 with Basic CCounter
P1 / P0 = 0.034 in terms of avg. CPU utilization (%)
P1 / P0 = 1.054 in terms of avg. memory utilization (%)
P1 / P0 = 2.159 in terms of elapsed time (s)
P1 / P0 = 0.073 in terms of est. CPU usage time (s)
P1 / P0 = 2.277 in terms of est. memory usage time (s)
The cost to run P1 was 0.01 CAD

User 3 with Stochastic CCounter
P1 / P0 = 0.01 in terms of avg. CPU utilization (%)
P1 / P0 = 1.138 in terms of avg. memory utilization (%)
P1 / P0 = 1.444 in terms of elapsed time (s)
P1 / P0 = 0.015 in terms of est. CPU usage time (s)
P1 / P0 = 1.643 in terms of est. memory usage time (s)
The cost to run P1 was 0.05 CAD

User 4 with Kalmann
P1 / P0 = 0.008 in terms of avg. CPU utilization (%)
P1 / P0 = 1.446 in terms of avg. memory utilization (%)
P1 / P0 = 1.337 in terms of elapsed time (s)
P1 / P0 = 0.011 in terms of est. CPU usage time (s)
P1 / P0 = 1.933 in terms of est. memory usage time (s)
The cost to run P2 was 0.16 CAD


Findings:
We verify that typically, par

### Stress Testing (PyTest, no longer necessary) - Consequtive and Simultaneous Submissions


Now, we do some performance tests. We now visit the live site and compare the results of doing 3 simultaneous submissions of the basic coloumb counter (forced P1) versus running them locally thrice, in 3 separate threads!

We already examined the utilization, now all I care about is the elapsed time!


In [7]:
import subprocess
import sys
from concurrent.futures import ThreadPoolExecutor

# Simulated function representing your pipeline submission
def run_external_script(script_name, arg1, arg2):
    """
    Executes the python script as a separate process.
    """
    # Use sys.executable to ensure we use the same environment's Python
    result = subprocess.run(
        [sys.executable, script_name, arg1, arg2], 
        capture_output=True, 
        text=True
    )
    return result.stdout

def test_triple_script_execution(workload=None):
    script_to_run = "standardized_evaluation_tool.py"

    # ARGUMENTS
    # Format: (arg1, arg2)
    if workload is None:
        workload = [
            ("user1", "dynamic"),
            ("user1", "dynamic"),
            ("user1", "dynamic")
        ]
    
    with ThreadPoolExecutor(max_workers=3) as executor:
        # Launching all 3 simultaneously
        futures = [
            executor.submit(run_external_script, script_to_run, args[0], args[1]) 
            for args in workload
        ]
        
        # Gathering results
        results = [f.result() for f in futures]

    # Verification
    assert len(results) == 3
    for output in results:
        print(f"Script output: {output}")

test_triple_script_execution()

Script output: 
Script output: {"error": false, "results_path": "w:\\Academia\\McMaster University\\Year 4\\00 Capstone\\Battery-Algorithm-Standardized-Testing-Tool\\Parallelization Workbench\\SETool_Python\\user1\\results\\results.zip", "Weighted_Error": 5669.928, "Test_Scores": [4966.925, 3696.596, 5390.368, 5177.895, 1174.117, 3696.596, 6245.049, 8751.939, 4883.597, 5133.581, 538.355, 185.912, 912.14, 1338.712, 1790.077, 2279.506, 10180.346, 11126.333], "All_Drive_Cycles_Average_RMSE": 4966.925, "All_Drive_Cycles_Average_MAE": 4966.923, "All_Drive_Cycles_Average_MAXE": 4968.669, "Complexity": "0,1,2"}

Script output: 


Compared to this, we tested with a brand new install and determined that the time was 5 minutes and 16 seconds. Examining the performance log, we note that each submission took roughly 44.124848772574175 seconds.
We note that parallelization, therefore, is very unsuited to working with P0 models, and thus, we move onto our final test of our complexity grader.


### Complexity Grader


Rcall that the complexity grader is "complexity_grader.py," it is responsible for grading a Python model on heuristics we have developed. It returns a score from 0-2, in ascending complexity. We called these "P0-P2" scores, which stand for "Necessary Parallelization Level." These levels are broadly categorized as follows:

These measures also have corresponding MATLAB complexities returned by the evaluator! They range from 0-9, and it returns a range with width of 3-4, such that P0 models are roughly 0,1,2,3 complex, P1 are roughly 4,5,6, and P2 are roughly 7,8,9. These grades are provided **after** each model is completely evaluated, so we will be using this metric later to demonstrate the correctness of our complexiy grader!

From the above, we know that the 4 models ought to return:

- 0,1,2
- 0,1,2
- 6,7,8
- 7,8,9

Therefore, the matchup should be:

- user 1 gets P0
- user 2 gets P0
- user 3 gets P1 or P2
- user 4 gets P2

We can verify the complexity grader is doing it's job by setting the models to run "dynamic"
This setting is enabled by default for user submissions.
We will not fully evaluate every model again, I'll interrupt each time when we can view the choice made by the grader in the log trace:


In [11]:
%run standardized_evaluation_tool.py "user1" "dynamic"

Loading test data...
Starting the blind modeling tool...
Zip extracted successfully.
Model file found!
Starting validation...
Starting validation process...
Validation passed!
Running test cycles...
Dynamically Adjusting Resource Allocation Schedule...
Task determined to be P0, with allocation time of 8 minutes


KeyboardInterrupt: 

In [12]:
%run standardized_evaluation_tool.py "user2" "dynamic"

Loading test data...
Starting the blind modeling tool...
Zip extracted successfully.
Model file found!
Starting validation...
Starting validation process...
Validation passed!
Running test cycles...
Dynamically Adjusting Resource Allocation Schedule...
Task determined to be P0, with allocation time of 8 minutes


KeyboardInterrupt: 

In [13]:
%run standardized_evaluation_tool.py "user3" "dynamic"

Loading test data...
Starting the blind modeling tool...
Zip extracted successfully.
Model file found!
Starting validation...
Starting validation process...
Validation passed!
Running test cycles...
Dynamically Adjusting Resource Allocation Schedule...
Task determined to be P1, with allocation time of 6 minutes
Scheduler Activated - Processing mode P1
Spinning up a cluster...
[2026-04-07 19:03:40,533][INFO    ][coiled] Fetching latest package priorities...
[2026-04-07 19:03:40,535][INFO    ][coiled.package_sync] Resolving your local .venv Python environment...
[2026-04-07 19:03:40,683][INFO    ][coiled.package_sync] Scanning 110 python packages...
[2026-04-07 19:03:40,926][INFO    ][coiled] Running pip check...
[2026-04-07 19:03:41,610][INFO    ][coiled] Validating environment...
[2026-04-07 19:03:46,196][INFO    ][coiled] Creating wheel for W:\Academia\McMaster University\Year 4\00 Capstone\Battery-Algorithm-Standardized-Testing-Tool\Parallelization Workbench\SETool_Python...
[2026-04

KeyboardInterrupt: 

In [14]:
%run standardized_evaluation_tool.py "user4" "dynamic"

Loading test data...
Starting the blind modeling tool...
Zip extracted successfully.
Model file found!
Starting validation...
Starting validation process...
Validation passed!
Running test cycles...
Dynamically Adjusting Resource Allocation Schedule...
Task determined to be P0, with allocation time of 6 minutes


KeyboardInterrupt: 

We notice that the complexity grader exercises correct heuristic judgement for 3 of 4 tests; we fail on the last example because model 4 involves an import from matlab files which raises its comlexity outside of the scope of Python code, with additional loops and a matrix diagonalization. This highlights the failings of a heuristics-based approach, but we are still satisfied that this simple method can reasonably detect when to swap between local and parallelized loads, with the worst case being a complex model run entirely on local, clogging up the CPU. In such a case, we have not yet devised strong measures to detect this, because we reason that it could also be a truly simple model run on a very large dataset. We ultimately fall back to cluster time limits (maximum of 12 hours, set in coiled) to prevent overuse of resources.
